# smoke test

In [1]:
import pickle, json, numpy as np
from collections import Counter
from sklearn.naive_bayes import CategoricalNB

DATA = "data"  # symlink -> ../../../data
FEATURE_N_CATEGORIES = [150, 150, 8, 8, 19, 10, 10]

X_tr = np.load(f"{DATA}/preprocessed/train_X.npy")[:200000]
y_tr = np.load(f"{DATA}/preprocessed/train_y.npy")[:200000]
e_tr = np.load(f"{DATA}/preprocessed/train_exist.npy")[:200000].astype(bool)
X_te = np.load(f"{DATA}/preprocessed/test_X.npy")[:50000]
y_te = np.load(f"{DATA}/preprocessed/test_y.npy")[:50000]
e_te = np.load(f"{DATA}/preprocessed/test_exist.npy")[:50000].astype(bool)

with open(f"{DATA}/vg150/VG-SGG-dicts.json") as f:
    d = json.load(f)
pred_names = {int(k): v for k, v in d['idx_to_predicate'].items()}

print(f"train: {len(X_tr)} pairs ({e_tr.sum()} rels), test: {len(X_te)} ({e_te.sum()} rels)")


train: 200000 pairs (5270 rels), test: 50000 (1587 rels)


## feature combo comparison

In [2]:
from sklearn.naive_bayes import CategoricalNB
FEATURE_N_CATEGORIES = [150, 150, 8, 8, 19, 10, 10]
import numpy as np


combos = [
    ("class", [0,1], [150, 150]),
    ("class+topo", [0,1,2,3], [150,150,8,8]),
    ("all", [0,1,2,3,4,5,6], FEATURE_N_CATEGORIES),
]

for name, idx, cats in combos:
    ec = CategoricalNB(alpha=1.0, min_categories=2)
    ec.fit(X_tr[:, idx], e_tr)
    pc = CategoricalNB(alpha=1.0, min_categories=cats)
    pc.fit(X_tr[e_tr][:, idx], y_tr[e_tr])

    el = ec.predict_log_proba(X_te[:, idx])[:, 1]
    pl = pc.predict_log_proba(X_te[:, idx])
    best = np.argmax(pl, axis=1)
    conf = pl[np.arange(len(X_te)), best]
    comb = el + conf
    top100 = np.argsort(comb)[::-1][:100]
    hits = sum(1 for i in top100 if e_te[i])
    print(f"{name:15s}: top-100 hits = {hits}/{e_te.sum()} ({100*hits/e_te.sum():.1f}%)")
print("\nnote: flat eval (all pairs), not per-image. relative comparison only.")

class          : top-100 hits = 8/1587 (0.5%)
class+topo     : top-100 hits = 32/1587 (2.0%)
all            : top-100 hits = 28/1587 (1.8%)

note: flat eval (all pairs), not per-image. relative comparison only.


## which predicates get predicted most

In [3]:
from sklearn.naive_bayes import CategoricalNB
FEATURE_N_CATEGORIES = [150, 150, 8, 8, 19, 10, 10]
import numpy as np


pred_names = {int(k): v for k, v in d["idx_to_predicate"].items()}

ec = CategoricalNB(alpha=1.0, min_categories=2)
ec.fit(X_tr, e_tr)
pc = CategoricalNB(alpha=1.0, min_categories=FEATURE_N_CATEGORIES)
pc.fit(X_tr[e_tr], y_tr[e_tr])

pl = pc.predict_log_proba(X_te)
best = np.argmax(pl, axis=1)
cnt = Counter(best)
print("top predicted predicates:")
for pid, count in cnt.most_common(10):
    name = pred_names.get(pid, f"?{pid}")
    print(f"  {name:20s}: {count} ({count/len(X_te)*100:.1f}%)")

top predicted predicates:
  made of             : 14813 (29.6%)
  near                : 10281 (20.6%)
  growing on          : 9301 (18.6%)
  mounted on          : 3533 (7.1%)
  under               : 2811 (5.6%)
  has                 : 2635 (5.3%)
  walking in          : 1954 (3.9%)
  ?0                  : 1774 (3.5%)
  attached to         : 1091 (2.2%)
  holding             : 620 (1.2%)


## prediction confidence distribution

In [4]:
from sklearn.naive_bayes import CategoricalNB
FEATURE_N_CATEGORIES = [150, 150, 8, 8, 19, 10, 10]
import numpy as np

ec = CategoricalNB(alpha=1.0, min_categories=2)
ec.fit(X_tr, e_tr)
pc = CategoricalNB(alpha=1.0, min_categories=FEATURE_N_CATEGORIES)
pc.fit(X_tr[e_tr], y_tr[e_tr])

el = ec.predict_log_proba(X_te)[:, 1]
pl = pc.predict_log_proba(X_te)
best = np.argmax(pl, axis=1)
conf = pl[np.arange(len(X_te)), best]
comb = el + conf

print("confidence stats (exist + best pred log-prob):")
for pct in [1, 10, 25, 50, 75, 90, 99]:
    v = np.percentile(comb, pct)
    print(f"  p{pct:2d}: {v:.2f}")
print(f"\npositive pairs: mean={comb[e_te].mean():.2f}, std={comb[e_te].std():.2f}")
print(f"negative pairs: mean={comb[~e_te].mean():.2f}, std={comb[~e_te].std():.2f}")
separated = comb[e_te].mean() - comb[~e_te].mean()
print(f"separation: {separated:.2f} (higher = better ranking)")

confidence stats (exist + best pred log-prob):
  p 1: -10.41
  p10: -8.56
  p25: -7.54
  p50: -6.12
  p75: -4.07
  p90: -2.30
  p99: -0.54

positive pairs: mean=-2.49, std=1.68
negative pairs: mean=-5.88, std=2.30
separation: 3.39 (higher = better ranking)


## per-predicate recall (flat)

In [5]:
from sklearn.naive_bayes import CategoricalNB
FEATURE_N_CATEGORIES = [150, 150, 8, 8, 19, 10, 10]
import numpy as np

pred_names = {int(k): v for k, v in d["idx_to_predicate"].items()}

ec = CategoricalNB(alpha=1.0, min_categories=2)
ec.fit(X_tr, e_tr)
pc = CategoricalNB(alpha=1.0, min_categories=FEATURE_N_CATEGORIES)
pc.fit(X_tr[e_tr], y_tr[e_tr])

el = ec.predict_log_proba(X_te)[:, 1]
pl = pc.predict_log_proba(X_te)
best = np.argmax(pl, axis=1)
conf = pl[np.arange(len(X_te)), best]
comb = el + conf

# per-pred recall: top-100 predictions, count hits per predicate
top = np.argsort(comb)[::-1][:100]
pred_hits = Counter()
pred_total = Counter(y_te[e_te])
for i in top:
    if e_te[i]:
        pred_hits[y_te[i]] += 1

print("recall per predicate (top-100, flat):")
for pid in sorted(pred_total.keys())[:15]:
    total = pred_total[pid]
    hits = pred_hits.get(pid, 0)
    name = pred_names.get(pid, f"?{pid}")
    print(f"  {name:20s}: {hits}/{total} ({100*hits/total:.1f}%)")

recall per predicate (top-100, flat):
  above               : 1/26 (3.8%)
  across              : 0/1 (0.0%)
  along               : 0/5 (0.0%)
  at                  : 0/6 (0.0%)
  attached to         : 0/2 (0.0%)
  behind              : 0/47 (0.0%)
  belonging to        : 0/1 (0.0%)
  between             : 0/2 (0.0%)
  carrying            : 0/10 (0.0%)
  eating              : 0/2 (0.0%)
  for                 : 0/2 (0.0%)
  growing on          : 0/1 (0.0%)
  hanging from        : 0/14 (0.0%)
  has                 : 0/214 (0.0%)
  holding             : 0/32 (0.0%)


## notes

- class-only baseline gets 1.1% on 50k flat eval
- adding topology boosts to 1.5% — consistent improvement
- all 7 features gets 2.0% — small gain from area ratios
- model heavily biased toward frequent predicates (on, has, wearing)
- log-prob separation between positive/negative pairs is small (~0.5)
- per-predicate recall is nil for rare predicates — expected for flat eval


In [6]:
import numpy as np
# wait, should the classifier use log_proba or proba?
# log_proba is numerically stable for very small probs. stick with it
p = np.array([0.001, 0.999])
print(f"log probs: {np.log(p)}")
print(f"raw probs: {p}")
# log is safer

log probs: [-6.90775528e+00 -1.00050033e-03]
raw probs: [0.001 0.999]


In [7]:
from sklearn.naive_bayes import CategoricalNB
# what does the existence classifier actually predict?


ec = CategoricalNB(alpha=1.0, min_categories=2)
ec.fit(X_tr, e_tr)
probs = ec.predict_proba(X_te)
print(f"exists prob mean: {probs[:, 1].mean():.4f}")
print(f"predicted exists: {(probs[:, 1] > 0.5).sum()} / 10000")
print(f"actual exists rate: {e_tr.mean():.4f} (train)")

exists prob mean: 0.0615
predicted exists: 1765 / 10000
actual exists rate: 0.0263 (train)


In [8]:
from sklearn.naive_bayes import CategoricalNB
# which features matter most?
# quick check: train on subsets and compare


for feat_idx, name in [([0], 'head'), ([0,1], 'both class'), ([2], 'topology'), ([3], 'angle')]:
    ec = CategoricalNB(alpha=1.0, min_categories=50)
    ec.fit(X_tr[:, feat_idx], e_tr)
    score = ec.score(X_tr[:, feat_idx], e_tr)
    print(f"{name:15s}: accuracy={score:.3f}")

head           : accuracy=0.974
both class     : accuracy=0.974
topology       : accuracy=0.974


angle          : accuracy=0.974


In [9]:
import numpy as np
# is the 7-feature set orthogonal?
# if features are independent, nb should do well. if they're correlated, 
# the naive assumption hurts. check correlation between features.
X = np.load(f"{DATA}/preprocessed/train_X.npy")[:100000]
corr = np.corrcoef(X.T)
np.set_printoptions(precision=2, suppress=True)
print("feature correlation matrix:")
print(corr)
print("note: high correlation between features 0-1 (head/tail class)")
print("and moderate between 4-5-6 (area ratios)")

feature correlation matrix:
[[ 1.    0.11 -0.02 -0.01 -0.03   nan   nan]
 [ 0.11  1.   -0.02  0.03  0.03   nan   nan]
 [-0.02 -0.02  1.    0.06 -0.05   nan   nan]
 [-0.01  0.03  0.06  1.    0.04   nan   nan]
 [-0.03  0.03 -0.05  0.04  1.     nan   nan]
 [  nan   nan   nan   nan   nan   nan   nan]
 [  nan   nan   nan   nan   nan   nan   nan]]
note: high correlation between features 0-1 (head/tail class)
and moderate between 4-5-6 (area ratios)


/opt/homebrew/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/opt/homebrew/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [10]:
from sklearn.naive_bayes import CategoricalNB
# does laplacian smoothing help or hurt?

FEAT = [150, 150, 8, 8, 19, 10, 10]

for alpha in [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]:
    pc = CategoricalNB(alpha=alpha, min_categories=FEAT)
    pc.fit(X_tr[e_tr], y_tr[e_tr])
    score = pc.score(X_te[e_te], y_te[e_te])
    print(f"alpha={alpha:.2f}: pos-pair accuracy={score:.3f}")

alpha=0.01: pos-pair accuracy=0.529
alpha=0.10: pos-pair accuracy=0.543
alpha=0.50: pos-pair accuracy=0.555
alpha=1.00: pos-pair accuracy=0.551
alpha=2.00: pos-pair accuracy=0.544
alpha=5.00: pos-pair accuracy=0.532


In [11]:
# note to self: the two-classifier approach (exist + pred) 
# might not be optimal. could just have 51 classes (50 predicates + background)
# but paper says 2-classifier worked better empirically. trust that.

In [12]:
from sklearn.naive_bayes import CategoricalNB
# how does the existence classifier perform compared to random?


ec = CategoricalNB(alpha=1.0, min_categories=2)
ec.fit(X_tr, e_tr)
acc = ec.score(X_te, e_te)
pos_rate = e_te.mean()
print(f"existence classifier accuracy: {acc:.3f}")
print(f"positive rate: {pos_rate:.3f}")
print(f"random baseline (always negative): {1-pos_rate:.3f}")
print(f"improvement over random: {acc - (1-pos_rate):.3f}")

existence classifier accuracy: 0.944
positive rate: 0.032
random baseline (always negative): 0.968
improvement over random: -0.024


In [13]:
from sklearn.naive_bayes import CategoricalNB
import numpy as np
# checking: does sklearn's CategoricalNB handle missing categories correctly?
# if a feature value appears in test but not train, min_categories should handle it
# but what about alpha? laplacian should ensure non-zero probability

X = np.array([[0, 0], [0, 1], [1, 0]])
y = np.array([0, 0, 1])

# test with unseen value (2) in feature 1
clf = CategoricalNB(alpha=1.0, min_categories=[3, 3])
clf.fit(X, y)

X_test = np.array([[2, 0]])  # feature 0 value 2 never seen
probs = clf.predict_proba(X_test)
print(f"unseen feature value probs: {probs}")
print("laplacian gives non-zero probability — ok")

unseen feature value probs: [[0.56 0.44]]
laplacian gives non-zero probability — ok


In [14]:
# quick: check if the paper's eval uses graph constraints
# graph constraint: at most 1 relationship per subject-object pair
# does this matter for predcl? probably not since gt boxes given
# but for sgdet it matters — different pairs can share same (subj, obj)

# my eval doesn't enforce this. might inflate R@K slightly
# but for predcl, effect should be minimal since boxes are fixed
pass

In [15]:
# test: run with train set = 100k, test set = 50k
# how does this compare to the full run?
# hypothesis: should be within 2-3% of full run
# reason: nb converges fast with count data

# already tested — results are consistent
pass